# 04 Feature Engineering

In this notebook, the preprocessed text data is transformed into numerical feature representations suitable for machine learning models.

Since machine learning algorithms cannot directly operate on raw text, textual documents must first be converted into vectors.

The main steps in this notebook are:

- splitting the dataset into training and test subsets,
- creating TF-IDF representations of the text,
- comparing feature spaces obtained from different preprocessing strategies,
- reducing the number of attributes using feature selection.

The resulting feature matrices will be used in the following notebooks for training and evaluating classification models.

In [2]:
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

In [3]:
DATA_PATH = Path("../processed/reuters_preprocessed.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5000, 9)


,file_name,author,split,text,text_length,word_count,clean_text_basic,clean_text_advanced,word_count_advanced
0,106247newsML.txt,AaronPressman,train,The Internet may be overflowing with new techn...,2009,319,the internet may be overflowing with new techn...,internet may overflowing new technology crime ...,198
1,120600newsML.txt,AaronPressman,train,The U.S. Postal Service announced Wednesday a ...,2604,416,the u s postal service announced wednesday a p...,u postal service announced wednesday plan boos...,258
2,120683newsML.txt,AaronPressman,train,Elementary school students with access to the ...,491,72,elementary school students with access to the ...,elementary school student access internet lear...,50
3,136958newsML.txt,AaronPressman,train,An influential Internet organisation has backe...,2907,463,an influential internet organisation has backe...,influential internet organisation backed away ...,275
4,137498newsML.txt,AaronPressman,train,An influential Internet organisation has backe...,2299,356,an influential internet organisation has backe...,influential internet organisation backed away ...,211


In [5]:
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2500, 9)
Test shape: (2500, 9)


In [6]:
y_train = train_df["author"]
y_test = test_df["author"]

print("Number of train labels:", len(y_train))
print("Number of test labels:", len(y_test))
print("Number of classes:", y_train.nunique())

Number of train labels: 2500
Number of test labels: 2500
Number of classes: 50


In [7]:
tfidf_basic = TfidfVectorizer(
    max_features=5000
)

X_train_basic = tfidf_basic.fit_transform(train_df["clean_text_basic"])
X_test_basic = tfidf_basic.transform(test_df["clean_text_basic"])

print("X_train_basic:", X_train_basic.shape)
print("X_test_basic:", X_test_basic.shape)

X_train_basic: (2500, 5000)
X_test_basic: (2500, 5000)


In [8]:
tfidf_advanced = TfidfVectorizer(
    max_features=5000
)

X_train_advanced = tfidf_advanced.fit_transform(train_df["clean_text_advanced"])
X_test_advanced = tfidf_advanced.transform(test_df["clean_text_advanced"])

print("X_train_advanced:", X_train_advanced.shape)
print("X_test_advanced:", X_test_advanced.shape)

X_train_advanced: (2500, 5000)
X_test_advanced: (2500, 5000)


In [9]:
print("Basic vocabulary size:", len(tfidf_basic.vocabulary_))
print("Advanced vocabulary size:", len(tfidf_advanced.vocabulary_))

Basic vocabulary size: 5000
Advanced vocabulary size: 5000


## TF-IDF feature extraction

In order to apply machine learning models to text data, the documents must first be converted into numerical vectors.

For this purpose, we used **TF-IDF (Term Frequency–Inverse Document Frequency)**, a widely used text representation method that assigns higher importance to words that are frequent in a given document but less frequent across the entire dataset.

Two separate TF-IDF representations were created:

- one based on **basic preprocessed text**
- one based on **advanced preprocessed text**

The training and test sets were kept separate according to the original dataset split. This is important because the dataset already provides a predefined train/test partition, which should be preserved.

Both vectorizers were limited to a maximum of **5000 features**, which helps control dimensionality while still retaining a large amount of textual information.

By comparing the resulting vocabulary sizes and matrix dimensions, we can later analyze how preprocessing affects the feature space and model performance.

In [10]:
selector_1000 = SelectKBest(score_func=chi2, k=1000)

X_train_1000 = selector_1000.fit_transform(X_train_advanced, y_train)
X_test_1000 = selector_1000.transform(X_test_advanced)

print("X_train_1000:", X_train_1000.shape)
print("X_test_1000:", X_test_1000.shape)

X_train_1000: (2500, 1000)
X_test_1000: (2500, 1000)


In [11]:
selector_3000 = SelectKBest(score_func=chi2, k=3000)

X_train_3000 = selector_3000.fit_transform(X_train_advanced, y_train)
X_test_3000 = selector_3000.transform(X_test_advanced)

print("X_train_3000:", X_train_3000.shape)
print("X_test_3000:", X_test_3000.shape)

X_train_3000: (2500, 3000)
X_test_3000: (2500, 3000)


In [12]:
feature_names = tfidf_advanced.get_feature_names_out()

scores = selector_1000.scores_

feature_scores = pd.DataFrame({
    "word": feature_names,
    "score": scores
})

feature_scores = feature_scores.sort_values(by="score", ascending=False)

feature_scores.head(20)

,word,score
1911,gm,800.529044
240,apple,588.407086
530,bre,518.553568
652,cargo,505.662140
822,cocoa,489.522215
940,conrail,478.479537
1677,fcc,442.690314
3840,russia,429.956231
835,colombia,426.803890
1917,gold,394.255990


## Feature selection using Chi-Square

Text datasets typically produce very high-dimensional feature spaces when using TF-IDF representations. While a large vocabulary may contain useful information, it can also introduce noise and increase computational complexity.

To address this issue, feature selection was applied using the **Chi-Square (χ²) statistical test**.

The Chi-Square test evaluates the relationship between each feature (word) and the target variable (author). Words that are strongly associated with particular authors receive higher scores.

Using this approach, two reduced feature sets were created:

- **Top 1000 features**
- **Top 3000 features**

These reduced feature spaces allow us to evaluate how classification models perform when using fewer attributes.

Comparing models trained on the full TF-IDF representation (5000 features) with models trained on reduced feature sets will help determine whether feature selection improves performance or computational efficiency.

## Most informative features

The Chi-Square feature selection method identifies words that are strongly associated with particular authors.

Several of the highest scoring terms include:

- gm
- apple
- cargo
- cocoa
- conrail
- fcc
- russia
- gold
- hong
- kong
- ibm
- shanghai

Many of these words correspond to company names, geographic locations, or economic terminology. This observation is consistent with the nature of Reuters news articles, which often focus on financial markets, international trade, and technology companies.

Because some journalists tend to specialize in particular topics or regions, these words can serve as useful indicators for distinguishing between authors.

Feature selection therefore helps identify informative words that contribute most to the classification task while reducing the dimensionality of the feature space.

In [13]:
import pickle
from pathlib import Path

FEATURE_DIR = Path("../processed/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

with open(FEATURE_DIR / "features.pkl", "wb") as f:
    pickle.dump({
        "X_train_full": X_train_advanced,
        "X_test_full": X_test_advanced,
        "X_train_3000": X_train_3000,
        "X_test_3000": X_test_3000,
        "X_train_1000": X_train_1000,
        "X_test_1000": X_test_1000,
        "y_train": y_train,
        "y_test": y_test
    }, f)

print("Feature matrices saved.")

Feature matrices saved.


## Summary of feature engineering

In this notebook, the cleaned text data was transformed into numerical representations suitable for machine learning algorithms.

TF-IDF was used to convert textual documents into vector representations, producing a feature space of up to **5000 attributes**.

To analyze the effect of dimensionality reduction, feature selection using the **Chi-Square statistical test** was applied. Two reduced feature sets were created containing the **top 3000** and **top 1000** most informative features.

These feature matrices will be used in the next stage of the project, where multiple classification algorithms will be trained and evaluated.

Comparing models trained on full and reduced feature sets will allow us to assess the impact of dimensionality on model performance and computational efficiency.